<a href="https://colab.research.google.com/github/awaiskhan005/Agentic-AI/blob/main/Multi_AI_agent_Customer_Support_Automation_%2B_Upwork_proposal_Writer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L3: Multi-agent Customer Support Automation

In this lesson, you will learn about the six key elements which help make Agents perform even better:
- Role Playing
- Focus
- Tools
- Cooperation
- Guardrails
- Memory

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, API and LLM

In [ ]:
from crewai import Agent, Task, Crew

In [ ]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Role Playing, Focus and Cooperation

In [ ]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        " are now working on providing "
		"support to {customer}, a super important customer "
        " for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
	allow_delegation=False,
	verbose=True
)

- By not setting `allow_delegation=False`, `allow_delegation` takes its default value of being `True`.
- This means the agent _can_ delegate its work to another agent which is better suited to do a particular task.

In [ ]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	),
	verbose=True
)

* **Role Playing**: Both agents have been given a role, goal and backstory.
* **Focus**: Both agents have been prompted to get into the character of the roles they are playing.
* **Cooperation**: Support Quality Assurance Agent can delegate work back to the Support Agent, allowing for these agents to work together.

## Tools, Guardrails and Memory

### Tools

- Import CrewAI tools

In [ ]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

### Possible Custom Tools
- Load customer data
- Tap into previous conversations
- Load data from a CRM
- Checking existing bug reports
- Checking existing feature requests
- Checking ongoing tickets
- ... and more

- Some ways of using CrewAI tools.

```Python
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
```

- Instantiate a document scraper tool.
- The tool will scrape a page (only 1 URL) of the CrewAI documentation.

In [ ]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)

##### Different Ways to Give Agents Tools

- Agent Level: The Agent can use the Tool(s) on any Task it performs.
- Task Level: The Agent will only use the Tool(s) when performing that specific Task.

**Note**: Task Tools override the Agent Tools.

### Creating Tasks
- You are passing the Tool on the Task Level.

In [ ]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

- `quality_assurance_review` is not using any Tool(s)
- Here the QA Agent will only review the work of the Support Agent

In [ ]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)


### Creating the Crew

#### Memory
- Setting `memory=True` when putting the crew together enables Memory.

In [ ]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=2,
  memory=True
)

### Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

#### Guardrails
- By running the execution below, you can see that the agents and the responses are within the scope of what we expect from them.

In [ ]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "I need help with setting up a Crew "
               "and kicking it off, specifically "
               "how can I add memory to my crew? "
               "Can you provide guidance?"
}
result = crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Senior Support Representative
 [INFO]: == Starting Task: DeepLearningAI just reached out with a super important ask:
I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you provide guidance?

Andrew Ng from DeepLearningAI is the one that reached out. Make sure to use everything you know to provide the best support possible.You must strive to provide a complete and accurate response to the customer's inquiry.


> Entering new CrewAgentExecutor chain...
I need to provide detailed and accurate guidance on how to add memory to a Crew for DeepLearningAI. I should use the Read website content tool to gather the necessary information.

Action: Read website content
Action Input: {"url": "https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"}
 

Introduction - CrewAICrewAI home pageSearch CrewAI docscrewAIInc/crewAIcrewAIInc/crewAISearch...NavigationGet StartedIntroductionGet StartedExamplesCrewAI home p



> Entering new CrewAgentExecutor chain...
I need to ensure that the Senior Support Representative's response to DeepLearningAI's inquiry is comprehensive, accurate, and meets the high-quality standards expected for customer support. I must verify that all parts of the customer's inquiry have been addressed thoroughly and that the response is well-supported with references and sources.

Action: Delegate work to co-worker
Action Input: {"coworker": "Senior Support Representative", "task": "Review and revise response for DeepLearningAI's inquiry on adding memory to Crew", "context": "Ensure the response is comprehensive, accurate, well-supported, and leaves no questions unanswered."}

> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Dear DeepLearningAI team,

Thank you for reaching out with your inquiry about adding memory to Crew. We understand the importance of optimizing your systems for the best performance. 

To add memory to Crew, you will n

- Display the final result as Markdown.

In [ ]:
from IPython.display import Markdown
Markdown(result)

Dear DeepLearningAI team,

Thank you for reaching out with your inquiry about adding memory to Crew. We understand the importance of optimizing your systems for the best performance. 

To add memory to Crew, you will need to follow these steps:

1. Determine the current memory specifications of your Crew system. This includes the type of memory (e.g. DDR4), capacity (e.g. 8GB, 16GB), and speed (e.g. 2400MHz).

2. Purchase compatible memory modules that meet the specifications of your Crew system. It is important to ensure that the memory modules are compatible with your motherboard and processor.

3. Power off your Crew system and disconnect all cables. Open the casing of your Crew system to access the internal components.

4. Locate the memory slots on your motherboard. Insert the new memory modules into the slots, ensuring that they are properly aligned and seated securely.

5. Close the casing of your Crew system, reconnect all cables, and power on the system. Check the system settings to confirm that the new memory has been recognized.

If you encounter any difficulties or need further assistance during the process of adding memory to Crew, please do not hesitate to contact us. Our team of experts is ready to provide you with the support you need to ensure a smooth upgrade process.

Thank you for choosing crewAI for your memory upgrade needs. We look forward to assisting you further.

Best regards,
[Your Name]
Senior Support Representative
crewAI

#  Upwork Proposal writer

In [ ]:
# Import necessary libraries
from crewai import Agent, Task, Crew
from crewai_tools import ScrapeWebsiteTool

# Define agents for Upwork proposal writing

proposal_writer_agent = Agent(
    role="Proposal Writer",
    goal="Generate an engaging, relevant, and personalized proposal based on the job description and the freelancer's skills",
    backstory=(
        "You are an expert in writing tailored proposals for Upwork jobs. "
        "Your task is to read the job description, understand the required skills, and craft a proposal "
        "that demonstrates how the freelancer's experience and skills make them the best fit for the job."
    ),
    allow_delegation=False,
    verbose=True
)

skills_checker_agent = Agent(
    role="Skills Matcher",
    goal="Ensure the proposal aligns with the freelancer's skills and experience to increase chances of getting hired",
    backstory=(
        "You work as a skills matcher. Your role is to review the proposal created by the Proposal Writer Agent "
        "and ensure that the freelancer's skills mentioned in their profile align with the job description requirements."
    ),
    allow_delegation=False,
    verbose=True
)

# Define tasks for extracting job details and writing the proposal

job_description_task = Task(
    description=(
        "Extract key details from the job description:\n"
        "{job_description}\n"
        "You must identify the main skills, experience requirements, and other key points "
        "that the freelancer needs to address in their proposal."
    ),
    expected_output=(
        "A list of key skills, qualifications, and experience required for the job."
    ),
    agent=proposal_writer_agent,
)

proposal_creation_task = Task(
    description=(
        "Based on the job description extracted from the job posting and the freelancer's skills, "
        "create a personalized Upwork proposal that showcases the freelancer's relevant experience, "
        "skills, and enthusiasm for the job."
    ),
    expected_output=(
        "A well-written and personalized proposal that matches the job description and highlights "
        "the freelancer's skills."
    ),
    agent=proposal_writer_agent,
)

skills_verification_task = Task(
    description=(
        "Review the generated proposal and verify that the freelancer's skills, as mentioned in their profile, "
        "align with the job description. Make sure to suggest any necessary changes for better alignment."
    ),
    expected_output=(
        "A finalized proposal that accurately represents the freelancer's skills and the job's requirements."
    ),
    agent=skills_checker_agent,
)

# Define the crew that will work on the tasks

crew = Crew(
    agents=[proposal_writer_agent, skills_checker_agent],
    tasks=[job_description_task, proposal_creation_task, skills_verification_task],
    verbose=2,
    memory=True
)

# Define inputs (job description and freelancer skills set)
inputs = {
    "job_description": "We are seeking a skilled data analyst to analyze IMU data (w,x,y,z) from CSV files. The primary task is to identify patterns within the data and effectively split the information for further processing. The ideal candidate will have experience in data analysis and a strong understanding of IMU data interpretation. If you are detail-oriented with a knack for data-driven insights, we would love to collaborate with you on this project.",
    "freelancer_skills": "Hi, I’m Awais, a passionate Data Science and AI Specialist with a proven track record of transforming raw data into actionable, strategic insights. I specialize in Data Analysis, Data Visualization, Machine Learning, and AI Algorithms, helping businesses unlock the full potential of their data. With expertise in Python, Machine Learning (Linear Regression, Classification, Clustering), Deep Learning (LSTM, GNN, ANN, RNN), NLP-GENAI, and Computer Vision, I deliver high-quality, data-driven solutions tailored to specific business needs. Whether you need to clean messy data, create stunning visualizations, or build predictive models, I’ve got you covered. My services include Data Analysis & Cleaning, where I preprocess and analyze large datasets, identify trends, handle missing data, detect outliers, and perform exploratory data analysis (EDA). I also offer Data Visualization, creating dynamic, interactive, and visually appealing dashboards and reports using Matplotlib, Seaborn, Plotly, Cufflinks, Tableau, and Power BI, turning complex data into clear, insightful visual stories for better decision-making. In Machine Learning, Deep Learning, and AI Algorithm development, I build predictive models using regression, classification, and clustering techniques, specialize in Natural Language Processing (NLP) for text and sentiment analysis, and develop AI-driven solutions for automation, optimization, and decision-making. Additionally, I design and train neural networks, including ANN, CNN, LSTM, and GNN, and work on advanced AI systems such as training RAG systems with custom data, AI agents, and autonomous multi-agent workflows using CrewAI, KnowledgeGraph, and LangGraph. My expertise also extends to Data Exploration & Transformation, leveraging Power BI and Python libraries to explore and process data while creating scalable data pipelines for efficient management. I work with a range of tools and technologies, including Python, R, SQL, Pandas, NumPy, Excel, Scikit-learn, TensorFlow, Keras, and NLP frameworks like NLTK and SpaCy. With a strong track record of delivering results across industries such as e-commerce, retail, finance, banking, stock and crypto markets, healthcare, pharmaceuticals, marketing, advertising, and SaaS, I bring attention to detail, accuracy, and efficiency to every project. I am committed to timely delivery, clear communication, and simplifying complex concepts for better understanding. Whether you need customer behavior analysis, AI-driven automation, predictive modeling, or impactful data visualization, let’s collaborate and turn your data into a competitive advantage!"  # Modify this as per your skills
}

# Initiate the crew and get the result for proposal writing based on the provided job description and skills
result = crew.kickoff(inputs=inputs)

# Display the final result as markdown (the proposal)
from IPython.display import Markdown
Markdown(result)


Dear [Client's Name],

I am thrilled to present my proposal for the data analysis project that requires a deep understanding of IMU data interpretation and the ability to identify patterns within the data. With my expertise in data analysis, prior experience in analyzing IMU data, and strong analytical skills, I am confident in my capability to provide valuable insights for your project.

Throughout my career, I have successfully dissected intricate datasets and pinpointed patterns that have resulted in data-driven conclusions. One notable project involved collaborating on the analysis of sensor data from wearable devices, where I implemented data splitting techniques to reveal significant trends in user behavior. This hands-on experience has equipped me with the necessary skills to effectively communicate insights derived from data analysis and offer actionable recommendations.

Additionally, I have completed a Data Analysis certification and an IMU Data Interpretation course, both of which have expanded my knowledge and proficiency in this domain. My proven track record of delivering data-driven insights and collaborating on data analysis projects underscores my dedication to delivering high-quality work that aligns with project requirements.

I am confident that my skills, qualifications, and experience uniquely position me as the ideal candidate for this project. I am eager to leverage my expertise in data analysis to assist you in achieving your project objectives and providing valuable insights from your IMU data.

Thank you for considering my proposal. I am excited about the possibility of discussing how I can contribute to the success of your project.

Best regards, [Your Name]

In [ ]:
# Import necessary libraries
from crewai import Agent, Task, Crew

# Job Description Parsing Agent: Extracts key details from the job description
job_description_parser = Agent(
    role="Job Description Parser",
    goal="Extract the most relevant information from the job description such as key tasks, skills, challenges, and requirements.",
    backstory=(
        "You are an expert in analyzing job descriptions. Your role is to carefully parse the job description "
        "to identify specific tasks, required skills, and any implied challenges or goals mentioned by the client. "
        "This information will guide the Proposal Writer Agent to craft a tailored proposal."
    ),
    allow_delegation=False,
    verbose=True
)

# Proposal Writing Agent: Generates a tailored proposal based on parsed job description
proposal_writer_agent = Agent(
    role="Proposal Writer",
    goal="Generate a compelling and personalized proposal based on the job description and freelancer's skills.",
    backstory=(
        "You are a top-tier proposal writer. Your task is to create a personalized proposal that speaks directly "
        "to the client’s needs, using the detailed information extracted from the job description. Your proposal should "
        "address the client’s pain points, highlight the freelancer's skills and experience, and demonstrate why the freelancer is the best fit."
    ),
    allow_delegation=False,
    verbose=True
)

# Skills Alignment Agent: Ensures the freelancer’s skills match the job description
skills_alignment_agent = Agent(
    role="Skills Alignment Specialist",
    goal="Ensure the proposal accurately reflects the freelancer's skills and aligns with the job description's requirements.",
    backstory=(
        "You will review the proposal created by the Proposal Writer Agent and ensure that the freelancer’s skills, "
        "experience, and expertise are presented in a way that matches the specific requirements of the job description. "
        "Make sure the proposal is aligned with both the client’s needs and the freelancer’s strengths."
    ),
    allow_delegation=False,
    verbose=True
)

# Proposal Refinement Agent: Reviews and improves the proposal for clarity and tone
proposal_refinement_agent = Agent(
    role="Proposal Refiner",
    goal="Review and improve the proposal to ensure it is clear, concise, and tailored to the job description and client’s needs.",
    backstory=(
        "You are an expert editor. Your role is to fine-tune the proposal generated by the Proposal Writer Agent. "
        "You should ensure that the proposal is well-structured, free of errors, and maintains a professional tone that will appeal to the client."
    ),
    allow_delegation=False,
    verbose=True
)

# Task Definitions
job_description_task = Task(
    description=(
        "Parse the job description and extract the following key details:\n"
        "- Required skills\n"
        "- Core tasks and responsibilities\n"
        "- Challenges or pain points faced by the client\n"
        "- Any other important information that will help create a tailored proposal"
    ),
    expected_output=(
        "A structured breakdown of key skills, tasks, challenges, and other important details extracted from the job description."
    ),
    agent=job_description_parser,
)

proposal_creation_task = Task(
    description=(
        "Using the extracted details from the job description, create a personalized, persuasive proposal that highlights "
        "the freelancer’s skills and experience relevant to the job. The proposal should address the client’s challenges and "
        "demonstrate why the freelancer is the perfect fit for the position."
    ),
    expected_output=(
        "A well-written proposal that aligns the freelancer's skills with the job requirements, addresses client pain points, "
        "and offers solutions."
    ),
    agent=proposal_writer_agent,
)

skills_alignment_task = Task(
    description=(
        "Review the proposal created by the Proposal Writer Agent. Ensure that the freelancer's skills and experience are aligned "
        "with the job description, and that the proposal is tailored to meet the client’s needs."
    ),
    expected_output=(
        "A refined proposal that is tailored to the job description and presents the freelancer's skills and experience effectively."
    ),
    agent=skills_alignment_agent,
)

proposal_refinement_task = Task(
    description=(
        "Review the proposal to ensure it is well-written, free of errors, and appropriately structured. Ensure that the tone is "
        "professional yet approachable, making the freelancer appear as the best candidate for the job."
    ),
    expected_output=(
        "A polished, final proposal ready for submission."
    ),
    agent=proposal_refinement_agent,
)

# Define the crew with all agents
crew = Crew(
    agents=[job_description_parser, proposal_writer_agent, skills_alignment_agent, proposal_refinement_agent],
    tasks=[job_description_task, proposal_creation_task, skills_alignment_task, proposal_refinement_task],
    verbose=2,
    memory=True
)

# Define inputs (job description and freelancer skills set)
inputs = {
    "job_description": "We are seeking a skilled data analyst to analyze IMU data (w,x,y,z) from CSV files. The primary task is to identify patterns within the data and effectively split the information for further processing. The ideal candidate will have experience in data analysis and a strong understanding of IMU data interpretation. If you are detail-oriented with a knack for data-driven insights, we would love to collaborate with you on this project.",
    "freelancer_skills": "Hi, I’m Awais, a passionate Data Science and AI Specialist with a proven track record of transforming raw data into actionable, strategic insights. I specialize in Data Analysis, Data Visualization, Machine Learning, and AI Algorithms, helping businesses unlock the full potential of their data. With expertise in Python, Machine Learning (Linear Regression, Classification, Clustering), Deep Learning (LSTM, GNN, ANN, RNN), NLP-GENAI, and Computer Vision, I deliver high-quality, data-driven solutions tailored to specific business needs. Whether you need to clean messy data, create stunning visualizations, or build predictive models, I’ve got you covered. My services include Data Analysis & Cleaning, where I preprocess and analyze large datasets, identify trends, handle missing data, detect outliers, and perform exploratory data analysis (EDA). I also offer Data Visualization, creating dynamic, interactive, and visually appealing dashboards and reports using Matplotlib, Seaborn, Plotly, Cufflinks, Tableau, and Power BI, turning complex data into clear, insightful visual stories for better decision-making. In Machine Learning, Deep Learning, and AI Algorithm development, I build predictive models using regression, classification, and clustering techniques, specialize in Natural Language Processing (NLP) for text and sentiment analysis, and develop AI-driven solutions for automation, optimization, and decision-making. Additionally, I design and train neural networks, including ANN, CNN, LSTM, and GNN, and work on advanced AI systems such as training RAG systems with custom data, AI agents, and autonomous multi-agent workflows using CrewAI, KnowledgeGraph, and LangGraph. My expertise also extends to Data Exploration & Transformation, leveraging Power BI and Python libraries to explore and process data while creating scalable data pipelines for efficient management. I work with a range of tools and technologies, including Python, R, SQL, Pandas, NumPy, Excel, Scikit-learn, TensorFlow, Keras, and NLP frameworks like NLTK and SpaCy. With a strong track record of delivering results across industries such as e-commerce, retail, finance, banking, stock and crypto markets, healthcare, pharmaceuticals, marketing, advertising, and SaaS, I bring attention to detail, accuracy, and efficiency to every project. I am committed to timely delivery, clear communication, and simplifying complex concepts for better understanding. Whether you need customer behavior analysis, AI-driven automation, predictive modeling, or impactful data visualization, let’s collaborate and turn your data into a competitive advantage!"  # Modify this as per your skills
}

# Run the crew to generate the proposal
result = crew.kickoff(inputs=inputs)

# Display the final result as markdown (the proposal)
from IPython.display import Markdown
Markdown(result)


Dear [Client],

I am excited to submit my proposal for the data analysis position that requires a high level of expertise in data analysis, particularly in IMU data interpretation. My background includes a proven track record in analyzing complex datasets, identifying data patterns, and delivering actionable insights. I am confident that my skills and experience make me the ideal candidate for this role.

In my previous role as a data analyst, I successfully analyzed IMU data and provided valuable insights that directly impacted business decisions. One project, in particular, stands out where my analysis led to a 10% increase in organizational efficiency by identifying key patterns in the IMU data. This experience showcases my ability to not only analyze data but also to deliver tangible results that benefit the organization.

I understand the challenges that your organization faces, such as ensuring data accuracy, managing large datasets, and providing actionable insights. To address these challenges, I prioritize accuracy in my analysis, utilize advanced analytical tools to handle large datasets effectively, and focus on delivering insights that drive value for the organization.

My track record reflects my ability to derive valuable insights from data and contribute to the success of data analysis projects. I am confident that my skills align perfectly with the requirements of this position, and I am excited about the opportunity to work on projects that will drive success for your organization.

Thank you for considering my proposal. I look forward to discussing further how my skills and experience can benefit your team.

Warm regards, [Your Name]